> 🔎 **원본(HD 1920×1080) 버전** — 입력 `clips_hd`, 결과 `auto_D3_eyes_open_hd.csv`. 240p 결과와 검출률 비교가 목적.

# D3_eyes_open — 무라벨(라벨링 없이) 획득

MediaPipe FaceLandmarker(478 랜드마크 + 블렌드셰이프)로 클립당 10프레임을 균등 샘플링해 양안 EAR(Soukupová & Čech 2016)를 계산하고, 얼굴 검출 프레임 중 EAR>임계값 비율(open_frac)로 클립 단위 눈뜸(1)/감음(0)/미상(-1)을 판정한다. 240x136 저해상도와 야간 IR에 대응해 3배 업스케일과 검출 신뢰도 완화(0.3)를 적용하고, 블렌드셰이프 eyeBlinkLeft/Right를 보조 신호로 기록해 EAR 판정과의 일치율로 자체 교차 검증한다. 임계값(0.21)은 성인 기준 시작값이며 검증 셀의 EAR 분포 쌍봉 골짜기로 재보정 가능하도록 원시값을 모두 CSV에 저장한다.

In [ ]:
# 설치 셀 — 처음 실행 후 protobuf/numpy 경고가 나오면: 런타임 → 세션 다시 시작 → 이어서 실행
!pip install -q mediapipe==0.10.14 av decord
!pip -q uninstall -y tensorflow  # mediapipe 0.10.14(protobuf4)와 TF(protobuf5) 충돌 방지 — 이 노트북은 TF 미사용

In [ ]:
# 공통 설정 — Drive 마운트 + 경로 + manifest
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import pandas as pd

BASE = Path('/content/drive/MyDrive/BabyMon/dataset')   # ← 업로드 위치
CLIPS = None
for cand in (BASE/'clips', BASE/'resized_2', BASE):
    if cand.is_dir() and next(cand.glob('*.mp4'), None):
        CLIPS = cand; break
assert CLIPS, 'mp4 폴더 없음 (clips/ 또는 resized_2/)'
assert (BASE/'manifest.csv').exists(), 'manifest.csv 를 BASE 에 업로드하세요'
WORK = Path('/content/work'); WORK.mkdir(exist_ok=True)
man = pd.read_csv(BASE/'manifest.csv')
print('clips:', CLIPS, len(list(CLIPS.glob("*.mp4"))), '개 / manifest:', len(man))

# ── HD(원본) 모드: clips_hd 를 우선 사용 ─────────────────────────
_hd = BASE/'clips_hd'
if _hd.is_dir() and next(_hd.glob('*.mp4'), None):
    CLIPS = _hd
    print('HD 모드: ', CLIPS, len(list(CLIPS.glob("*.mp4"))), '개 (1920x1080)')
else:
    raise SystemExit('clips_hd 없음 — D:\\carved\\dataset\\clips_hd 를 Drive 의 dataset/clips_hd 로 업로드하세요')

## D3_eyes_open — 눈 뜸 자동 라벨 (무라벨)

**방법**: MediaPipe **FaceLandmarker**(478 랜드마크, 눈/홍채 정밀화 + 블렌드셰이프)로 프레임별 얼굴 랜드마크를 추정하고, Soukupová & Čech (2016)의 **EAR(Eye Aspect Ratio)** 을 양안 평균으로 계산한다.

클립당 10프레임 균등 샘플링 → 얼굴 검출 프레임의 EAR 통계로 클립 라벨 결정:
- `eyes_open = 1` : open_frac(EAR > 0.21 프레임 비율) ≥ 0.5
- `eyes_open = 0` : 그 외 (눈 감음)
- `eyes_open = -1` : 얼굴 검출 2프레임 미만 → 미상(후속 단계에서 제외/별도 처리)

보조 신호로 블렌드셰이프 `eyeBlinkLeft/Right`(0=뜸, 1=감음)도 기록해 EAR 판정과의 일치율로 교차 검증한다.

240x136 저해상도 + 야간 IR 대응: **3배 업스케일**(INTER_CUBIC), 얼굴 검출 신뢰도 0.3으로 완화. IR 흑백 프레임도 3채널 RGB로 디코딩되므로 그대로 입력 가능. 임계값 0.21은 성인 기준 시작값이며, 아래 검증 셀의 EAR 분포(쌍봉 골짜기)를 보고 재보정할 것 — 원시값(ear_med, ear_p90, open_frac, blink_med)을 모두 저장하므로 라벨 재계산은 CSV만으로 가능하다.

**출처**
- Soukupová & Čech, *Real-Time Eye Blink Detection using Facial Landmarks*, CVWW 2016 — [Semantic Scholar](https://www.semanticscholar.org/paper/Real-Time-Eye-Blink-Detection-using-Facial-Soukupov%C3%A1-Cech/4fa1ba3531219ca8c39d8749160faf1a877f2ced)
- Grishchenko et al., *Attention Mesh: High-fidelity Face Mesh Prediction in Real-time*, 2020 — [arXiv](https://arxiv.org/pdf/2006.10962)
- Moezzi et al., *Classification of Infant Sleep–Wake States from Natural Overnight In-Crib Sleep*, WACV Workshops 2025 — [CVF](https://openaccess.thecvf.com/content/WACV2025W/CV4Small/papers/Moezzi_Classification_of_Infant_Sleep-Wake_States_from_Natural_Overnight_In-Crib_Sleep_WACVW_2025_paper.pdf)

In [ ]:
# 모델 초기화: MediaPipe Tasks FaceLandmarker (블렌드셰이프 포함)
# 참고: Tasks API 실패 시 레거시 mp.solutions.face_mesh(refine_landmarks=True)로 대체 가능
import cv2, urllib.request, numpy as np, pandas as pd
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from decord import VideoReader

MODEL_PATH = WORK / 'face_landmarker.task'
if not MODEL_PATH.exists():
    urllib.request.urlretrieve(
        'https://storage.googleapis.com/mediapipe-models/face_landmarker/'
        'face_landmarker/float16/1/face_landmarker.task', str(MODEL_PATH))

opts = mp_vision.FaceLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=str(MODEL_PATH)),
    running_mode=mp_vision.RunningMode.IMAGE,
    num_faces=1,
    min_face_detection_confidence=0.3,   # 저해상도/야간 IR 대응으로 완화
    output_face_blendshapes=True)
landmarker = mp_vision.FaceLandmarker.create_from_options(opts)

# EAR용 눈 랜드마크 인덱스 (MediaPipe 468 토폴로지, P1..P6 = 수평 2점 + 수직 4점)
LEFT_EYE  = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]
EAR_THR, SCALE, N_FRAMES, MIN_FACE = 0.21, 3, 10, 2
print('FaceLandmarker ready:', MODEL_PATH)

In [ ]:
def _ear(pts, idx):
    v1 = np.linalg.norm(pts[idx[1]] - pts[idx[5]])
    v2 = np.linalg.norm(pts[idx[2]] - pts[idx[4]])
    h  = np.linalg.norm(pts[idx[0]] - pts[idx[3]])
    return (v1 + v2) / (2 * h + 1e-6)

def process_clip(fname):
    out = {'file': fname, 'n_sampled': 0, 'n_face': 0, 'ear_med': np.nan,
           'ear_p90': np.nan, 'blink_med': np.nan, 'open_frac': np.nan,
           'eyes_open': -1, 'error': ''}
    try:
        vr = VideoReader(str(CLIPS / fname), num_threads=2)
        ii = np.linspace(0, len(vr) - 1, min(N_FRAMES, len(vr))).astype(int)
        frames = vr.get_batch(ii).asnumpy()          # (n, H, W, 3) RGB
    except Exception as e:
        out['error'] = f'read:{e}'; return out
    out['n_sampled'] = len(frames)
    ears, blinks = [], []
    for f in frames:
        img = cv2.resize(f, None, fx=SCALE, fy=SCALE, interpolation=cv2.INTER_CUBIC)
        res = landmarker.detect(mp.Image(image_format=mp.ImageFormat.SRGB,
                                         data=np.ascontiguousarray(img)))
        if not res.face_landmarks:
            continue
        h, w = img.shape[:2]
        pts = np.array([[l.x * w, l.y * h] for l in res.face_landmarks[0]])
        ears.append((_ear(pts, LEFT_EYE) + _ear(pts, RIGHT_EYE)) / 2)
        try:  # 블렌드셰이프 키 이름은 버전에 따라 다를 수 있어 방어적으로 처리
            bs = {b.category_name: b.score for b in res.face_blendshapes[0]}
            blinks.append((bs['eyeBlinkLeft'] + bs['eyeBlinkRight']) / 2)
        except Exception:
            pass
    if ears:
        ears = np.array(ears)
        out.update(n_face=len(ears), ear_med=float(np.median(ears)),
                   ear_p90=float(np.percentile(ears, 90)),
                   open_frac=float((ears > EAR_THR).mean()))
        if blinks:
            out['blink_med'] = float(np.median(blinks))
        if len(ears) >= MIN_FACE:            # 얼굴이 충분히 보일 때만 판정
            out['eyes_open'] = int(out['open_frac'] >= 0.5)
    return out

print(process_clip(man.iloc[0]['file']))  # smoke test

In [ ]:
# 배치 루프 — BASE/'auto_D3_eyes_open_hd.csv' 에 증분 저장 (기처리 파일 건너뜀)
import time
OUT_CSV = BASE / 'auto_D3_eyes_open_hd.csv'
N_CLIPS = 1000

done = set(pd.read_csv(OUT_CSV)['file']) if OUT_CSV.exists() else set()
sample = man.sample(min(N_CLIPS, len(man)), random_state=42)['file'].tolist()
todo = [f for f in sample if f not in done]
print(f'표본 {len(sample)}개 중 처리 대상 {len(todo)}개 (기처리 {len(sample) - len(todo)}개 건너뜀)')

buf, t0 = [], time.time()
for i, fname in enumerate(todo, 1):
    buf.append(process_clip(fname))
    if i % 50 == 0 or i == len(todo):
        pd.DataFrame(buf).to_csv(OUT_CSV, mode='a', index=False,
                                 header=not OUT_CSV.exists())
        buf = []
        el = time.time() - t0
        print(f'{i}/{len(todo)}  경과 {el:.0f}s  ({el / i:.2f}s/clip)')
print('완료:', OUT_CSV)

In [ ]:
# 검증: 분포 확인 + EAR-블렌드셰이프 일치율 + 극단 사례 프레임
import matplotlib.pyplot as plt

df = pd.read_csv(OUT_CSV).drop_duplicates('file')
det = df[df['n_face'] >= MIN_FACE]
print(f'총 {len(df)}개 | 판정 가능(얼굴검출) {len(det)}개 ({len(det) / max(len(df), 1):.0%})')

C_BLUE, C_ORANGE, C_GRAY = '#3B6BB0', '#C07A2E', '#9AA0A6'
fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
ax[0].hist(det['ear_med'].dropna(), bins=40, color=C_BLUE)
ax[0].axvline(EAR_THR, color=C_GRAY, ls='--')
ax[0].set_title(f'EAR 중앙값 (점선 = 임계값 {EAR_THR})')
ax[1].hist(det['blink_med'].dropna(), bins=40, color=C_BLUE)
ax[1].set_title('blendshape blink 중앙값 (1=감음)')
cnt = df['eyes_open'].value_counts().reindex([1, 0, -1], fill_value=0)
bars = ax[2].bar(['open(1)', 'closed(0)', 'unknown(-1)'], cnt.values,
                 color=[C_BLUE, C_ORANGE, C_GRAY])
ax[2].bar_label(bars); ax[2].set_title('클립 라벨 분포')
for a in ax:
    a.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()

both = det.dropna(subset=['blink_med'])
if len(both):
    agree = ((both['ear_med'] > EAR_THR) == (both['blink_med'] < 0.4)).mean()
    print(f'EAR vs blendshape 판정 일치율: {agree:.0%}  (낮으면 임계값 재보정 필요)')

def show_frames(files, title):
    fig, axes = plt.subplots(1, len(files), figsize=(2.8 * len(files), 2.4))
    for a, f in zip(np.atleast_1d(axes), files):
        try:
            vr = VideoReader(str(CLIPS / f)); a.imshow(vr[len(vr) // 2].asnumpy())
        except Exception as e:
            a.text(0.5, 0.5, str(e)[:30], ha='center')
        a.set_title(str(f)[-24:], fontsize=7); a.axis('off')
    fig.suptitle(title); plt.tight_layout(); plt.show()

show_frames(det.nlargest(3, 'ear_med')['file'].tolist(), '극단 사례: EAR 최대 (눈 뜸 예상)')
show_frames(det.nsmallest(3, 'ear_med')['file'].tolist(), '극단 사례: EAR 최소 (눈 감음 예상)')